# Colab Training Notebook: Text-to-SVG Adapter v5



In [1]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q datasets trl transformers accelerate peft bitsandbytes pandas lxml scikit-learn


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 22.0 MB/s eta 0:00:00


In [2]:
import json
import os
import random
import re
import time
import warnings
import gc
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

_IS_COLAB = os.path.exists("/content")
if not _IS_COLAB:
    raise RuntimeError("This notebook is designed for Google Colab only.")

DRIVE_PROJECT_DIR = "ColabProjects/qwen_svg_lora_train_v5"  # NEW project dir
_DRIVE_BASE = f"/content/drive/MyDrive/{DRIVE_PROJECT_DIR}"
os.makedirs(_DRIVE_BASE, exist_ok=True)

_DATA_CANDIDATES = [
    Path("/content/dl-spring-2026-svg-generation"),
    Path("/content"),
    Path.cwd() / "dl-spring-2026-svg-generation",
    Path.cwd(),
]
for _candidate in _DATA_CANDIDATES:
    if (_candidate / "train.csv").exists():
        _DATA_DIR = str(_candidate)
        break
else:
    _DATA_DIR = str(_DATA_CANDIDATES[-1])

# ── Previous project dir (where checkpoint-9100 lives) ───────────────────
_PREV_DRIVE_BASE = "/content/drive/MyDrive/ColabProjects/qwen_svg_lora_train_v3"

CONFIG = {
    # Model
    "model_name":                   "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "max_seq_length":               2048,   # ← DOUBLED (was 1024). Critical for longer SVGs.
    "max_new_tokens":               1800,   # ← Updated to match (was 900)

    # LoRA (keep same r/alpha so the checkpoint loads cleanly)
    "lora_r":                       64,
    "lora_alpha":                   128,
    "lora_dropout":                 0.05,

    # Data
    "max_competition_samples":      50000,
    "max_external_per_source":      7000,
    "eval_size":                    0.02,
    "min_svg_chars":                300,    # ← NEW: filter trivially short SVGs

    # Smoke test
    "smoke_test_mode":              False,
    "smoke_train_samples":          256,
    "smoke_max_seq_length":         512,

    # Training
    "num_train_epochs":             5,      # 
    "per_device_train_batch_size":  1,
    "gradient_accumulation_steps":  16,
    "learning_rate":                3e-5,  # ← Lower: fine-tuning from strong ckpt (was 1.5e-4)
    "warmup_steps":                 20,     # ← Shorter warmup (was 50)
    "weight_decay":                 0.01,
    "logging_steps":                20,
    "save_steps":                   300,
    "save_total_limit":             2,

    # Resumption — point at checkpoint-9100 in the OLD project dir
    "resume_if_checkpoint_exists":  True,
    "resume_checkpoint_path":       f"{_PREV_DRIVE_BASE}/checkpoints/checkpoint-9100",

    # Paths
    "competition_data_dir":         _DATA_DIR,
    "output_dir":                   f"{_DRIVE_BASE}/checkpoints",
    "adapter_dir":                  f"{_DRIVE_BASE}/adapter",
    "run_info_path":                f"{_DRIVE_BASE}/run_info.json",
}

if CONFIG["smoke_test_mode"]:
    CONFIG["max_competition_samples"] = min(CONFIG["max_competition_samples"], CONFIG["smoke_train_samples"])
    CONFIG["max_seq_length"]          = min(CONFIG["max_seq_length"], CONFIG["smoke_max_seq_length"])
    CONFIG["save_steps"]              = min(CONFIG["save_steps"], 25)
    CONFIG["warmup_steps"]            = min(CONFIG["warmup_steps"], 5)

if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Google Drive is not mounted.")
    
# Verify checkpoint exists
_ckpt = Path(CONFIG["resume_checkpoint_path"])
if _ckpt.exists():
    print(f"✓ checkpoint-9100 found at {_ckpt}")
else:
    print(f"⚠️  checkpoint-9100 NOT found at {_ckpt}")
    print("   Options:")
    print("   1. Update resume_checkpoint_path to the correct path")
    print("   2. Set resume_if_checkpoint_exists=False to start fresh")

print(f"\nmax_seq_length      : {CONFIG['max_seq_length']} tokens")
print(f"learning_rate       : {CONFIG['learning_rate']}")
print(f"resume_checkpoint   : {CONFIG['resume_checkpoint_path']}")
print(f"output_dir          : {CONFIG['output_dir']}")
print(f"adapter_dir         : {CONFIG['adapter_dir']}")


✓ checkpoint-9100 found at /content/drive/MyDrive/ColabProjects/qwen_svg_lora_train_v3/checkpoints/checkpoint-9100

max_seq_length      : 2048 tokens
learning_rate       : 3e-05
resume_checkpoint   : /content/drive/MyDrive/ColabProjects/qwen_svg_lora_train_v3/checkpoints/checkpoint-9100
output_dir          : /content/drive/MyDrive/ColabProjects/qwen_svg_lora_train_v5/checkpoints
adapter_dir         : /content/drive/MyDrive/ColabProjects/qwen_svg_lora_train_v5/adapter


In [5]:
ALLOWED_TAGS = {
    "svg","g","path","rect","circle","ellipse","line","polyline","polygon",
    "defs","use","symbol","clipPath","mask","linearGradient","radialGradient",
    "stop","text","tspan","title","desc","style","pattern","marker","filter",
}

FALLBACK_SVG = (
    "<svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'>"
    "<rect x='32' y='32' width='192' height='192' rx='16' fill='#d1d5db'/>"
    "<circle cx='128' cy='128' r='48' fill='#6b7280'/></svg>"
)


def is_valid_svg(svg: str) -> bool:
    if not svg or len(svg) > 16_000:
        return False
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        return False
    path_count = 0
    for el in root.iter():
        tag = el.tag.split("}")[-1] if "}" in el.tag else el.tag
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
    return path_count <= 256


def has_minimum_visual_content(svg: str, min_chars: int = 300) -> bool:
    """Filter out trivially short/empty SVGs that add noise to training."""
    return len(svg) >= min_chars


def clean_svg(svg: str) -> str:
    svg = svg.strip()
    if not svg.startswith("<svg"):
        m = re.search(r"<svg", svg, re.IGNORECASE)
        if m:
            svg = svg[m.start():]
        else:
            return FALLBACK_SVG
    if "xmlns" not in svg[:200]:
        svg = svg.replace("<svg", "<svg xmlns='http://www.w3.org/2000/svg'", 1)
    return svg


def truncate_svg(svg: str, max_len: int = 15_800) -> str:
    if len(svg) <= max_len:
        return svg
    cut = svg.rfind("<", 0, max_len)
    return svg[:cut] + "</svg>"


In [6]:
data_dir = Path(CONFIG["competition_data_dir"])
train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")

assert list(test_df.columns) == ["id", "prompt"]
print("train rows:", len(train_df))
print("test rows:", len(test_df))


train rows: 50000
test rows: 1000


In [7]:
print("Validating and filtering competition SVGs ...")
pool_size = min(len(train_df), CONFIG["max_competition_samples"] * 3)
pool = train_df.sample(pool_size, random_state=SEED).reset_index(drop=True)

pool["_valid"] = pool["svg"].apply(is_valid_svg)
pool["_rich"]  = pool["svg"].apply(
    lambda s: has_minimum_visual_content(s, CONFIG["min_svg_chars"])
)
valid_pool = pool[pool["_valid"] & pool["_rich"]].drop(columns=["_valid", "_rich"]).reset_index(drop=True)

# Show SVG length stats so we can see what we're training on
svg_lens = valid_pool["svg"].str.len()
print(f"SVG length  — mean: {svg_lens.mean():.0f}  median: {svg_lens.median():.0f}  "
      f"p90: {svg_lens.quantile(0.9):.0f}  max: {svg_lens.max():.0f}")

comp_data = valid_pool.sample(
    min(CONFIG["max_competition_samples"], len(valid_pool)),
    random_state=SEED,
)[["prompt", "svg"]].reset_index(drop=True)
print("usable training samples:", len(comp_data))


Validating and filtering competition SVGs ...
SVG length  — mean: 2569  median: 2153  p90: 5131  max: 15937
usable training samples: 49025


In [8]:
# Enable the external dataset for more diversity
ACTIVE_SOURCES = []

DATASET_CATALOG = {
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
}


def _pick_first(example, keys):
    for k in keys:
        if k in example and example[k]:
            v = str(example[k]).strip()
            if v:
                return v
    return ""


def normalise_row(ex, prompt_fields, svg_fields):
    prompt = _pick_first(ex, prompt_fields)
    svg = _pick_first(ex, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}


external_parts = []
for src in ACTIVE_SOURCES:
    cfg = DATASET_CATALOG[src]
    ds = load_dataset(src, split=cfg["split"])
    if len(ds) > CONFIG["max_external_per_source"]:
        ds = ds.shuffle(seed=SEED).select(range(CONFIG["max_external_per_source"]))
    ds = ds.map(
        lambda ex: normalise_row(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
    )
    ds = ds.filter(
        lambda x: bool(x["prompt"]) and bool(x["svg"])
            and is_valid_svg(x["svg"])
            and has_minimum_visual_content(x["svg"], CONFIG["min_svg_chars"])
    )
    ext_df = ds.to_pandas()
    print(f"  {src}: {len(ext_df)} samples after filtering")
    external_parts.append(ext_df)

if external_parts:
    all_data = pd.concat([comp_data] + external_parts, ignore_index=True)\
                 .sample(frac=1, random_state=SEED).reset_index(drop=True)
else:
    all_data = comp_data.copy()

print("total training samples:", len(all_data))


total training samples: 49025


In [9]:
# ── IMPROVED system prompt: explicit visual fidelity guidance ──────────────
SYSTEM_PROMPT = (
    "You are an expert SVG artist and code generator. "
    "Given a natural-language description, output ONLY valid, visually accurate SVG code "
    "that faithfully represents the described scene or object. "
    "Requirements:\n"
    "- Root element: <svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'>\n"
    "- Use appropriate colors, shapes, and proportions to match the description visually\n"
    "- Fill the 256x256 canvas meaningfully — avoid large empty areas\n"
    "- Use multiple shapes (rect, circle, ellipse, path, polygon) layered for detail\n"
    "- Use fill and stroke colors that match the described appearance\n"
    "- Output SVG directly — no explanation, no markdown fences, no extra text"
)

# ── SVG truncation limit increased to match new token budget ───────────────
# At 2048 tokens total, ~1700 tokens are available for the SVG response.
# At ~3.5 chars/token for SVG code, that's ~5950 chars.
# We use 5500 as a safe ceiling.
_SVG_TRAIN_MAX_LEN = 5500  # was 2200


def format_sft_text(prompt: str, svg: str) -> str:
    svg = clean_svg(svg)
    svg = truncate_svg(svg, max_len=_SVG_TRAIN_MAX_LEN)
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{svg}<|im_end|>"
    )


all_data["text"] = all_data.apply(lambda r: format_sft_text(r["prompt"], r["svg"]), axis=1)

# Show text length distribution
txt_lens = all_data["text"].str.len()
print(f"Text length — mean: {txt_lens.mean():.0f}  median: {txt_lens.median():.0f}  "
      f"p90: {txt_lens.quantile(0.9):.0f}  max: {txt_lens.max():.0f}")

from sklearn.model_selection import train_test_split
train_texts, eval_texts = train_test_split(
    all_data["text"].tolist(),
    test_size=CONFIG["eval_size"],
    random_state=SEED,
)
train_hf = Dataset.from_dict({"text": train_texts})
eval_hf = Dataset.from_dict({"text": eval_texts})
print("train hf:", len(train_hf), "eval hf:", len(eval_hf))


Text length — mean: 3162  median: 2910  p90: 5374  max: 6800
train hf: 48044 eval hf: 981


In [10]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],  # 2048
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

tokenizer.model_max_length = CONFIG["max_seq_length"]
tokenizer.truncation_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def clamp_dataset_to_context(ds, split_name: str):
    def _clamp(batch):
        tokenized = tokenizer(
            batch["text"],
            add_special_tokens=False,
            truncation=True,
            max_length=CONFIG["max_seq_length"],
        )
        return {
            "text": tokenizer.batch_decode(
                tokenized["input_ids"],
                skip_special_tokens=False,
                clean_up_tokenization_spaces=False,
            ),
            "n_tokens": [len(ids) for ids in tokenized["input_ids"]],
        }
    ds = ds.map(_clamp, batched=True, batch_size=64, num_proc=1, desc=f"Clamping {split_name} texts")
    toks = ds["n_tokens"]
    print(f"{split_name} — max_tokens: {max(toks)}  mean_tokens: {sum(toks)/len(toks):.0f}")
    return ds.remove_columns(["n_tokens"])


train_hf = clamp_dataset_to_context(train_hf, "train")
eval_hf  = clamp_dataset_to_context(eval_hf,  "eval")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Clamping train texts (num_proc=1):   0%|          | 0/48044 [00:00<?, ? examples/s]

train — max_tokens: 2048  mean_tokens: 1587


Clamping eval texts (num_proc=1):   0%|          | 0/981 [00:00<?, ? examples/s]

eval — max_tokens: 2048  mean_tokens: 1598


In [11]:
import time, shutil, os, threading

def sync_loop():
    adapter_src  = CONFIG["adapter_dir"]
    ckpt_src     = CONFIG["output_dir"]
    while True:
        try:
            if os.path.exists(adapter_src):
                print(f"[{time.strftime('%H:%M:%S')}] Adapter synced ✓")
            if os.path.exists(ckpt_src):
                print(f"[{time.strftime('%H:%M:%S')}] Checkpoints exist ✓")
        except Exception as e:
            print(f"[{time.strftime('%H:%M:%S')}] Sync error: {e}")
        time.sleep(600)

t = threading.Thread(target=sync_loop, daemon=True)
t.start()
print("Background sync thread started ✓")


Background sync thread started ✓


In [ ]:
from transformers import TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTTrainer

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["adapter_dir"], exist_ok=True)

if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()


class SaveAdapterToDriveCallback(TrainerCallback):
    """Saves LoRA adapter to Drive at every checkpoint."""

    def __init__(self, adapter_dir: str):
        self.adapter_dir = adapter_dir

    def on_save(self, args, state, control, **kwargs):
        model = kwargs.get("model")
        if model is None:
            return
        step = state.global_step
        step_dir = os.path.join(self.adapter_dir, f"step_{step}")
        os.makedirs(step_dir, exist_ok=True)
        model.save_pretrained(step_dir)
        model.save_pretrained(self.adapter_dir)   # overwrite canonical
        print(f"\n[AdapterCallback] Step {step}: adapter saved")


training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["save_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    load_best_model_at_end=False,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataloader_num_workers=0,
)

# ── Smart checkpoint resume ───────────────────────────────────────────────
latest_checkpoint = None
if CONFIG.get("resume_if_checkpoint_exists", False):
    # First try to find a checkpoint in the NEW output_dir (previous v5 run)
    latest_checkpoint = get_last_checkpoint(CONFIG["output_dir"])
    if latest_checkpoint:
        print(f"Resuming from latest v5 checkpoint: {latest_checkpoint}")
    else:
        # Fall back to the explicitly named checkpoint (checkpoint-9100)
        explicit = CONFIG["resume_checkpoint_path"]
        if os.path.exists(explicit):
            latest_checkpoint = explicit
            print(f"Resuming from checkpoint-9100: {latest_checkpoint}")
        else:
            print(f"⚠️  checkpoint-9100 not found at {explicit} — starting fresh")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_hf,
    eval_dataset=eval_hf,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
    callbacks=[SaveAdapterToDriveCallback(adapter_dir=CONFIG["adapter_dir"])],
)

print(f"Training starts: {len(train_hf):,} samples  "
      f"| seq_len={CONFIG['max_seq_length']}  "
      f"| lr={CONFIG['learning_rate']}  "
      f"| epochs={CONFIG['num_train_epochs']}")
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
print(f"\nTraining done in {(time.time() - t0)/60:.1f} min")
print("Final train loss:", train_result.training_loss)


[12:31:01] Adapter synced ✓
[12:31:01] Checkpoints exist ✓
Resuming from latest v5 checkpoint: /content/drive/MyDrive/ColabProjects/qwen_svg_lora_train_v5/checkpoints/checkpoint-12012


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/48044 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/981 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training starts: 48,044 samples  | seq_len=2048  | lr=3e-05  | epochs=5


	eval_steps: 300 (from args) != 100 (from trainer_state.json)
	save_steps: 300 (from args) != 100 (from trainer_state.json)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48,044 | Num Epochs = 5 | Total steps = 15,015
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 73,859,072 of 1,617,573,376 (4.57% trained)


Step,Training Loss,Validation Loss
12100,0.236892,0.234979


[12:41:01] Adapter synced ✓
[12:41:01] Checkpoints exist ✓
[12:51:01] Adapter synced ✓
[12:51:01] Checkpoints exist ✓
[13:01:01] Adapter synced ✓
[13:01:01] Checkpoints exist ✓

[AdapterCallback] Step 12100: adapter saved


In [ ]:
os.makedirs(CONFIG["adapter_dir"], exist_ok=True)
trainer.save_model(CONFIG["adapter_dir"])
tokenizer.save_pretrained(CONFIG["adapter_dir"])

run_info = {
    "model_name": CONFIG["model_name"],
    "max_seq_length": CONFIG["max_seq_length"],
    "competition_data_dir": CONFIG["competition_data_dir"],
    "output_dir": CONFIG["output_dir"],
    "adapter_dir": CONFIG["adapter_dir"],
}
with open(CONFIG["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)

print("Adapter saved to:", CONFIG["adapter_dir"])
print("Run info saved to:", CONFIG["run_info_path"])


## Merge adapter into base model (for Kaggle inference)



In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED_MODEL_DIR = f"{_DRIVE_BASE}/merged_model"
BASE_MODEL_NAME  = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_DIR      = CONFIG["adapter_dir"]

os.makedirs(MERGED_MODEL_DIR, exist_ok=True)
gc.collect()
torch.cuda.empty_cache()

print("Loading base model (fp16) ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading adapter ...")
merged_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Merging and saving (this takes ~10 min) ...")
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
merged_tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("Merged model saved to:", MERGED_MODEL_DIR)
